# 10 · Advanced Recursion

A second, deeper pass on the **pick / not-pick** pattern from notebook 07. Here the focus is on *seeing* the recursion tree, understanding pruning, and choosing the right return type for each kind of question.

**Topics covered**
1. Power set: bitmask vs. recursion
2. Subsequences with sum `k` — print **all**
3. Subsequences with sum `k` — print **one** (early exit)
4. Subsequences with sum `k` — **count**
5. Binary strings of length `n` with no consecutive 1s
6. Edge case: zeros and negatives break the `total > k` pruning

## The four "answer shapes" of a recursive search

| Question | Function returns | At the base case |
|---|---|---|
| Find **all** answers | nothing (appends to `result`) | `result.append(subset.copy())` |
| Find **any one** answer | `True` / `False` | `return True` when found |
| **Count** the answers | an `int` | `return 1` (valid) or `return 0` (invalid) |
| Find the **best** answer | the best value so far | `return value` |

Almost every recursion problem fits one of these four shapes. Decide the shape first, then write the pick / not-pick body.

## 1. Power set (all subsets)

`n` elements → `2ⁿ` subsets. At every index we make **two choices**, so the recursion tree has `2ⁿ` leaves.

```
                        index 0
                 pick 1 /      \ skip 1
                   [1]            []
           pick 2 /   \ skip 2   /   \
              [1,2]   [1]      [2]    []
               ...     ...      ...   ...
```

**Time:** O(n · 2ⁿ) · **Space:** O(n) recursion stack (plus the output)

In [1]:
# Method 1: bitmask — each number 0 … 2^n-1 is a "yes/no" mask over the elements
nums = [1, 2, 3]
n = len(nums)
result = []
for mask in range(1 << n):
    subset = [nums[i] for i in range(n) if mask & (1 << i)]
    result.append(subset)
print("Bitmask  :", result)

Bitmask  : [[], [1], [2], [1, 2], [3], [1, 3], [2, 3], [1, 2, 3]]


In [2]:
# Method 2: recursion (pick first, then not pick)
def power_set(index, subset, nums, result):
    if index >= len(nums):                       # base case: every element has been decided
        result.append(subset.copy())             # copy → later pops must not change what we stored
        return
    subset.append(nums[index])                   # choice 1: PICK
    power_set(index + 1, subset, nums, result)
    subset.pop()                                 # undo the pick (BACKTRACK)
    power_set(index + 1, subset, nums, result)   # choice 2: NOT PICK

result = []
power_set(0, [], [1, 2, 3], result)
print("Recursion:", result)
assert sorted(map(tuple, result)) == sorted(map(tuple, [[], [1], [2], [1, 2], [3], [1, 3], [2, 3], [1, 2, 3]]))

Recursion: [[1, 2, 3], [1, 2], [1, 3], [1], [2, 3], [2], [3], []]


## 2. All subsequences with sum `k`

Carry a running `total`. Stop a branch as soon as it is decided:
- `total == k` → success, store a copy
- `total > k` → fail early (**pruning** — saves a lot of work)
- `index == n` → fail (no elements left)

Example: `nums = [2, 3, 6, 4, 7]`, `k = 7` → `[3, 4]` and `[7]`.

In [3]:
def all_with_sum(index, total, subset, nums, k, result):
    if total == k:
        result.append(subset.copy())
        return
    if total > k or index >= len(nums):          # prune / out of elements
        return
    subset.append(nums[index])
    all_with_sum(index + 1, total + nums[index], subset, nums, k, result)   # pick
    subset.pop()
    all_with_sum(index + 1, total, subset, nums, k, result)                 # not pick

result = []
all_with_sum(0, 0, [], [2, 3, 6, 4, 7], 7, result)
print(result)                                    # [[3, 4], [7]]

[[3, 4], [7]]


### Watch the recursion tree
The same function, but every call prints itself with indentation proportional to its depth. This makes the pruning visible: branches that hit `total > k` stop immediately.

In [4]:
def traced(index, total, subset, nums, k, depth=0):
    pad = "  " * depth
    if total == k:
        print(f"{pad}✅ total={total} → found {subset}")
        return
    if total > k:
        print(f"{pad}✂️  total={total} > k → prune")
        return
    if index >= len(nums):
        print(f"{pad}❌ out of elements (total={total})")
        return
    print(f"{pad}index={index}, total={total}, subset={subset}")
    subset.append(nums[index])
    traced(index + 1, total + nums[index], subset, nums, k, depth + 1)       # pick
    subset.pop()
    traced(index + 1, total, subset, nums, k, depth + 1)                     # not pick

traced(0, 0, [], [2, 3, 4], 5)

index=0, total=0, subset=[]
  index=1, total=2, subset=[2]
    ✅ total=5 → found [2, 3]
    index=2, total=2, subset=[2]
      ✂️  total=6 > k → prune
      ❌ out of elements (total=2)
  index=1, total=0, subset=[]
    index=2, total=3, subset=[3]
      ✂️  total=7 > k → prune
      ❌ out of elements (total=3)
    index=2, total=0, subset=[]
      ❌ out of elements (total=4)
      ❌ out of elements (total=0)


## 3. Print any ONE subsequence with sum `k`

Once an answer is found there is no need to explore the rest of the tree. Return **`True`** the moment one is found and let every caller pass that `True` upward:

```python
if pick_branch(...): return True     # found in the pick branch → stop here
...
return not_pick_branch(...)          # otherwise the answer is whatever the other branch finds
```

Note we only `pop()` after the pick branch **fails**: if it succeeded we want to keep the answer inside `subset`.

In [5]:
def one_with_sum(index, total, subset, nums, k, result):
    if total == k:
        result.append(subset.copy())
        return True
    if total > k or index >= len(nums):
        return False
    subset.append(nums[index])
    if one_with_sum(index + 1, total + nums[index], subset, nums, k, result):
        return True                              # early exit, don't explore anything else
    subset.pop()
    return one_with_sum(index + 1, total, subset, nums, k, result)

result = []
print(one_with_sum(0, 0, [], [2, 3, 6, 4, 7], 11, result), result)   # True [[2, 3, 6]]
result = []
print(one_with_sum(0, 0, [], [2, 3, 6, 4, 7], 100, result), result)  # False []

True [[2, 3, 6]]
False []


## 4. COUNT the subsequences with sum `k`

We do not need the subsets themselves, only how many there are. So the function **returns a number**: `pick_count + not_pick_count`. No list, no backtracking `pop()`, and memory use is just the recursion stack.

This version also shows why DP exists: many `(index, total)` pairs get computed more than once (try printing a counter to see it). Memoising `(index, total)` is exactly what the dynamic-programming version does.

In [6]:
def count_with_sum(index, total, nums, k):
    if total == k:
        return 1
    if total > k or index >= len(nums):
        return 0
    pick = count_with_sum(index + 1, total + nums[index], nums, k)
    not_pick = count_with_sum(index + 1, total, nums, k)
    return pick + not_pick

print(count_with_sum(0, 0, [2, 3, 6, 4, 7], 9))      # 3 → [2,3,4], [2,7], [3,6]

3


## 5. Binary strings of length `n` with no two consecutive 1s

Build the string left to right. At each position we can **always** place `0`; we may place `1` only if the **previous** character was not `1`. A boolean `can_place_one` carries that information.

```
n = 3:  000 001 010 100 101      (5 strings)
```
The counts follow the Fibonacci sequence: n=1→2, n=2→3, n=3→5, n=4→8, n=5→13.

In [7]:
class Solution:
    def solve(self, index, can_place_one, chars, result):
        if index >= len(chars):
            result.append("".join(chars))
            return
        chars[index] = "0"                                   # option 1: always allowed
        self.solve(index + 1, True, chars, result)
        if can_place_one:                                    # option 2: only after a 0
            chars[index] = "1"
            self.solve(index + 1, False, chars, result)
            chars[index] = "0"                               # backtrack

    def generate(self, n):
        result = []
        self.solve(0, True, ["0"] * n, result)
        return result

print(Solution().generate(4))
print([len(Solution().generate(n)) for n in range(1, 8)])   # Fibonacci-like: 2, 3, 5, 8, 13, 21, 34

['0000', '0001', '0010', '0100', '0101', '1000', '1001', '1010']
[2, 3, 5, 8, 13, 21, 34]


## 6. Edge case: pruning with `total > k` needs **non-negative, non-zero** numbers

If the array can contain **negative numbers**, a running total above `k` can still come back down to `k`, so pruning is wrong (`[5, -2]` reaches 3 only after passing 5). Likewise, with **zeros**, returning as soon as `total == k` skips the subsets that add zeros afterwards (`[3]` and `[3, 0]` both sum to 3).

Fix: remove both early returns and only judge the total when **every element has been decided** (`index == n`).

In [8]:
def count_no_prune(index, total, nums, k):
    if index == len(nums):                       # decide only at the end
        return 1 if total == k else 0
    return (count_no_prune(index + 1, total + nums[index], nums, k) +
            count_no_prune(index + 1, total, nums, k))

for nums, k in ([5, -2, 1, 0], 3), ([3, 0], 3):
    brute = sum(1 for mask in range(1 << len(nums))
                if sum(nums[i] for i in range(len(nums)) if mask & (1 << i)) == k)   # trusted bitmask count
    pruned, exact = count_with_sum(0, 0, nums, k), count_no_prune(0, 0, nums, k)
    print(f"nums={nums}, k={k} → pruned: {pruned} | no pruning: {exact} | brute force: {brute}")
    assert exact == brute and pruned != brute      # pruning gives the WRONG answer here

nums=[5, -2, 1, 0], k=3 → pruned: 0 | no pruning: 2 | brute force: 2
nums=[3, 0], k=3 → pruned: 1 | no pruning: 2 | brute force: 2


---
### ✅ Key takeaways
- Pick the **answer shape** first: all → append, one → `True/False`, count → return a number.
- `subset.copy()` when storing, `subset.pop()` to backtrack.
- **Prune** early when the problem allows (non-negative numbers), but know when pruning is invalid.
- Counting recursions repeat the same `(index, total)` states → that is the doorway to **dynamic programming**.